# FIFA World Cup — Experimentation & Causal Analytics

Three experiments on open World Cup data:
1. **VAR causal impact** — interrupted time-series + match-level Poisson GLM
2. **Shootout first-kicker advantage** — natural experiment + power analysis
3. **32-team vs 48-team format** — designed A/B test via Monte Carlo

In [ ]:
import sys, os
sys.path.append(os.path.abspath('../src'))
import pandas as pd, numpy as np
from data_loader import download, tournament_rates
download()
tournament_rates(mens_only=True)[
    ['tournament_name','year','matches','penalty_rate','post_var']].tail(8)

## Module 1 — Did VAR change penalty rates?

In [ ]:
from module1_var import fit_match_level, fit_its
ml = fit_match_level()
print(f"Match-level IRR: {ml['irr']:.2f}x (95% CI {ml['irr_ci'][0]:.2f}-{ml['irr_ci'][1]:.2f}), p={ml['p_value']:.4f}")
its = fit_its('penalty_rate')
print(f"Tournament-level: p={its['p_value']:.3f} (underpowered)")

In [ ]:
from viz import fig_var
from IPython.display import Image
Image(fig_var())

## Module 2 — Does kicking first win the shootout?

In [ ]:
from module2_shootout import run_test
r = run_test()
print(f"First kicker won {r['win_rate']*100:.1f}% (n={r['n_shootouts']}), p={r['p_value']:.3f}")

In [ ]:
from viz import fig_shootout
Image(fig_shootout())

## Module 3 — 32-team vs 48-team A/B test

In [ ]:
from elo import compute_ratings
compute_ratings().head(8).round(0)

In [ ]:
from module3_format_ab import run_experiment, power_n
print(f'Power analysis: {power_n()} sims/arm; using floor of 1000.')
exp = run_experiment()
labels = {'champion_elo':'Champion Elo','top4_seed_won':'Top-4 seed won',
          'knockout_upsets':'Knockout upsets','champion_seed_rank':'Champion rank'}
pd.DataFrame([{'Metric':labels[k],'32-team':round(v['mean_A_32team'],3),
  '48-team':round(v['mean_B_48team'],3),"Cohen's d":round(v['cohens_d'],2),
  'p':round(v['p_value'],4),'Sig':'Yes' if v['significant'] else 'no'}
  for k,v in exp['results'].items()])

In [ ]:
from viz import fig_format
Image(fig_format(exp))